# New York Has Twelve Little Indias

DBSCAN on every active restaurant the health department lists as Indian. Run `fetch_week1.py day4` first.

## 1 · Settings

In [ ]:
DAY      = 4
PROMPT   = "Clusters"
SLUG     = "clusters-nyc-indian"
NOTEBOOK = "day04_clusters_nyc_indian.ipynb"
TITLE    = "New York Has\nTwelve Little\nIndias"
CREDITS  = "Restaurants: NYC Department of Health restaurant inspections, active restaurants listed as Indian. Boundary: US Census Bureau."

## 2 · Load

In [ ]:
import geopandas as gpd, matplotlib.pyplot as plt, numpy as np, pandas as pd
from sklearn.cluster import DBSCAN
from mapkit import poster, publish
from mapkit.paths import PROCESSED

food = gpd.read_file(PROCESSED / "nyc_indian.geojson")
city = gpd.read_file(PROCESSED / "places_six.geojson").query("city == 'nyc'")

## 3 · Analysis, every number on the map comes from here

In [ ]:
EPS_M, MIN_SIZE = 400, 5      # at least five restaurants, each about a five-minute walk from the next
m = food.to_crs(32618)
food["cluster"] = DBSCAN(eps=EPS_M, min_samples=MIN_SIZE).fit_predict(np.c_[m.geometry.x, m.geometry.y])
sizes = food[food.cluster >= 0].groupby("cluster").size().sort_values(ascending=False)
in_clusters = int(sizes.sum())

# Names by where each cluster's center falls, checked by hand against its street addresses
PLACES = {"Curry Hill": (-73.986, 40.745), "Midtown West": (-73.986, 40.762), "Midtown East": (-73.972, 40.755), "East Village": (-73.985, 40.727),
          "Greenwich Village": (-74.001, 40.730), "Tribeca": (-74.008, 40.716), "Upper West Side": (-73.966, 40.801),
          "Richmond Hill": (-73.829, 40.689), "Richmond Hill south": (-73.828, 40.694), "Jackson Heights": (-73.892, 40.749), "Bedford-Stuyvesant": (-73.951, 40.681), "Floral Park": (-73.711, 40.737)}
centers = food[food.cluster >= 0].groupby("cluster").geometry.apply(lambda g: (g.x.mean(), g.y.mean()))
names = {c: min(PLACES, key=lambda p: (PLACES[p][0] - xy[0]) ** 2 + (PLACES[p][1] - xy[1]) ** 2) for c, xy in centers.items()}
assert len(set(names.values())) == len(names), "two clusters got the same name"
print(f"Indian restaurants  {len(food)}")
print(f"Clusters            {len(sizes)}")
print(f"In a cluster        {in_clusters} ({in_clusters / len(food) * 100:.0f}%)")
print(f"Dosa in the name    {int(food.dosa_named.astype(bool).sum())}")
for c, n in sizes.items():
    print(f"  {names[c]:22} {n}")

## 4 · Map

In [ ]:
SAFFRON, LAND, CHILI, INK = "#f2ae21", "#fff3d6", "#b3201f", "#3a120c"
fig, ax = poster.page(SAFFRON)
city.to_crs(3857).plot(ax=ax, color=LAND, edgecolor=INK, lw=0.7, zorder=1)
food_m = food.to_crs(3857)
ax.set_axis_off()
fx0, fy0, fx1, fy1 = poster.fit_frame(ax, -74.27, 40.49, -73.69, 40.92)

# ripples around each cluster, thinning as they spread
for c in sizes.index:
    pts = food_m[food_m.cluster == c]
    hull = pts.geometry.union_all().convex_hull
    for grow, lw, alpha in [(180, 1.25, 1), (420, 0.7, 0.85), (720, 0.38, 0.7), (1080, 0.2, 0.55)]:
        gpd.GeoSeries([hull.buffer(grow * 1.32)], crs=3857).plot(ax=ax, facecolor="none", edgecolor=CHILI, lw=lw, alpha=alpha, zorder=3)
    gpd.GeoSeries([hull.buffer(180 * 1.32)], crs=3857).plot(ax=ax, facecolor=CHILI, alpha=0.22, lw=0, zorder=2)
food_m[food_m.cluster < 0].plot(ax=ax, color=INK, markersize=8, alpha=0.55, lw=0, zorder=4)
food_m[food_m.cluster >= 0].plot(ax=ax, color=CHILI, markersize=17, lw=0, zorder=5)
ax.set_xlim(fx0, fx1); ax.set_ylim(fy0, fy1)

# Manhattan's clusters get a stacked list out in the water, with hairlines back to each one
W = fx1 - fx0
manhattan = ["Upper West Side", "Midtown West", "Midtown East", "Curry Hill", "East Village", "Greenwich Village", "Tribeca"]
by_name = {v: k for k, v in names.items()}
ys = np.linspace(fy1 - (fy1 - fy0) * 0.33, fy1 - (fy1 - fy0) * 0.60, len(manhattan))
for y, name in zip(ys, manhattan):
    c = by_name[name]
    cx, cy = food_m[food_m.cluster == c].geometry.union_all().centroid.coords[0]
    ax.plot([fx0 + W * 0.335, cx], [y, cy], color=INK, lw=0.5, zorder=6)
    ax.text(fx0 + W * 0.29, y, name, color=INK, fontproperties=poster.font(poster.POPPINS, 12.5), ha="right", va="center", zorder=7)
    ax.text(fx0 + W * 0.297, y, f"{sizes[c]}", color=CHILI, fontproperties=poster.font(poster.POPPINS_BOLD, 12.5), ha="left", va="center", zorder=7)

def tag(name, dx, dy, ha="left"):
    c = by_name[name]
    cx, cy = food_m[food_m.cluster == c].geometry.union_all().centroid.coords[0]
    ax.text(cx + W * dx, cy + W * dy, f"{name.replace(' south', '')} {sizes[c]}", color=INK, fontproperties=poster.font(poster.POPPINS_BOLD, 11.5), ha=ha, va="center", zorder=7)

tag("Jackson Heights", 0.035, 0.03)
tag("Bedford-Stuyvesant", 0.045, -0.03, ha="left")
tag("Richmond Hill", 0.04, -0.03)
tag("Floral Park", -0.03, 0.035, ha="right")
tag("Richmond Hill south", 0.04, 0.03)

fig.text(0.06, 0.965, "New York Has", color=INK, fontproperties=poster.font(poster.YATRA, 42), va="top")
fig.text(0.06, 0.915, "Twelve Little", color=CHILI, fontproperties=poster.font(poster.YATRA, 42), va="top")
fig.text(0.06, 0.865, "Indias", color=CHILI, fontproperties=poster.font(poster.YATRA, 42), va="top")
fig.text(0.06, 0.805, f"{len(food)} Indian restaurants. {in_clusters} sit\nin tight clusters of five or more,\neach a five-minute walk from the next.", color=INK,
         fontproperties=poster.font(poster.POPPINS, 12.5), va="top", linespacing=1.4)
poster.credit(fig, CREDITS, path=poster.POPPINS, name_path=poster.POPPINS_BOLD, color="#5e3320", name_color=INK, size=8.4)

## 5 · Export

In [ ]:
publish.save(fig, day=DAY, slug=SLUG, prompt=PROMPT, title=TITLE.replace("\n", " "),
             subtitle=f"{len(food)} Indian restaurants, {len(sizes)} clusters", series="city", tool="Python",
             notebook=NOTEBOOK, credits=CREDITS)